In [1]:
import ollama
import json
from Error import Error

from pydantic import BaseModel

class ErrorGetterFormat(BaseModel):
        error_type: str
        severity: str
        description: str

class ErrorListFormat(BaseModel):
        errors: list[ErrorGetterFormat]

class SuggestionGetterFormat(BaseModel):
        suggestion: str

class SuggestionListFormat(BaseModel):
        suggestions: list[SuggestionGetterFormat]

prompt_config_path = "prompts.json"
host = "http://localhost:11434" # Ollama binds to port 11434 by default
model = "llama3:latest"
available_error_types = "OVERFLOW_ERROR, ROUND_OFF_ERROR, INFINITE_LOOP_ERROR," \
                "MODIFY_PARAMETER_VARIABLE_ERROR, OFF_BY_ONE_ERROR, ARITHMETIC_ERROR," \
                "BOUNDS_ERROR, UNINITIALIZED_ARRAY_ERROR, SQUELCH_EXCEPTION_ERROR," \
                "MAGIC_NUMBER_ERROR, DANGLING_ELSE_ERROR, OTHERS"

# input from the changes in mined repo's pull request
code = "" \
        "counter = 0:\n" \
        "while counter < 5:\n" \
        "\tprint(counter)" 

In [2]:
# get the prompts from the json file

try:
    with open(prompt_config_path, "r", encoding="utf-8") as file:
        prompts = json.load(file)

        system_prompt_error = prompts['system_prompt_error']
        print(f"System prompt error fetched: {system_prompt_error}")

        system_prompt_suggestion = prompts['system_prompt_suggestion']
        print(f"System prompt suggestion fetched: {system_prompt_suggestion}")

        get_error_prompt = prompts['get_error_prompt']
        print(f"Get error prompt fetched: {get_error_prompt}")

        suggestion_prompt = prompts['suggestion_prompt']
        print(f"Suggestion prompt fetched: {suggestion_prompt}")

        print(f"\n'{prompt_config_path}' successfully loaded!")
        
except FileNotFoundError:
    print(f"Error: The file '{prompt_config_path}' was not found.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")

client = ollama.Client(host=host)

System prompt error fetched: You are an expert software engineer doing python code review. When given a code sample and a list of possible errors, find if any of the errors exist in the code. You can only choose up to 2 error types. Always return your response as a Python list of strings in this format: Error Type | Error severity (low/medium/high) | Error description. If no errors are found return: NONE
System prompt suggestion fetched: You are an expert software engineer doing python code review. Suggest a fix for an error when you are given an error type, error severity level, and error description. You should only write one suggestion
Get error prompt fetched: Here is the code sample {code} and the list of possible errors {available_error_types}
Suggestion prompt fetched: Here is the error type: {error_type}, severity level: {severity} and description: {error_description}. This is the code snippet: {code}.

'prompts.json' successfully loaded!


In [3]:
# get error prompt

def request_error(get_error_prompt: str) -> str:
    get_error_prompt = get_error_prompt.format(
        code = code, # the code in string format containing the changes in a specific pull request mined from the repo
        available_error_types = available_error_types
    )

    get_error_response = client.chat(
        model = model,
        messages = [
            {"role": "system", "content": system_prompt_error},
            {"role": "user", "content": get_error_prompt}
        ],
        format = ErrorListFormat.model_json_schema()
    )

    error_response = get_error_response["message"]["content"]
    return error_response

def parse_error_response(error_response: str) -> list[Error]:
    # will parse the response from llm
    if error_response == "NONE":
        return []
    
    error_response = json.loads(error_response)
    
    parsed_errors = error_response['errors'] # end result or parsing the error response from LLM: a list of error information in consistent format

    # Create a list of Error objects. Assign error type, severity level, and error description generated from llm
    errors = []

    for idx, error_info in enumerate(parsed_errors):
        error_type = error_info['error_type']
        error_severity_level = error_info['severity']
        error_description = error_info['description']

        error = Error(error_type, error_severity_level, error_description)
        errors.append(error)

    return errors

error_response = request_error(get_error_prompt)
errors = parse_error_response(error_response)

for error in errors:
    print(error.error_type)
    print(error.error_severity_level)
    print(error.error_description)
    print()

INFINITE_LOOP_ERROR
high
The while loop will run indefinitely because there is no increment operation on the counter variable.

BOUNDS_ERROR
low
The loop condition only checks if the counter is less than 5, but does not consider what happens when the counter becomes equal to 5. 



In [4]:
# get fix suggestion

def request_suggestion(error: Error) -> Error:
    error_type = error.get_error_type()
    severity = error.get_error_severity_level()
    error_description = error.get_error_description()

    global suggestion_prompt

    suggestion_prompt = suggestion_prompt.format(
        error_type = error_type,
        severity = severity,
        error_description = error_description,
        code = code
    )

    suggestion_response = client.chat(
        model = model,
        messages = [
            {"role": "system", "content": system_prompt_suggestion},
            {"role": "user", "content": suggestion_prompt}
        ],
        format=SuggestionListFormat.model_json_schema()
    )

    suggestion_response = suggestion_response["message"]["content"]

    suggestion_response_parsed = json.loads(suggestion_response)['suggestions'][0]['suggestion']

    error.set_fix_suggestion(suggestion_response_parsed)

    return error

def get_all_fix_suggestions(errors: list[Error]) -> list[Error]:
    for idx, error in enumerate(errors):
        errors[idx] = request_suggestion(error)

    return errors

errors = get_all_fix_suggestions(errors)

for error in errors:
    print(error.error_type)
    print(error.error_severity_level)
    print(error.error_description)
    print(error.fix_suggestion)
    print()

INFINITE_LOOP_ERROR
high
The while loop will run indefinitely because there is no increment operation on the counter variable.
Add an increment operator to the counter variable inside the while loop to ensure it increments and eventually reaches the condition's threshold, thus avoiding the infinite loop. Here's the corrected code snippet:

counter = 0;
while counter < 5:
    counter += 1;
    print(counter)

BOUNDS_ERROR
low
The loop condition only checks if the counter is less than 5, but does not consider what happens when the counter becomes equal to 5. 
Add an increment operation to the counter variable within the while loop, e.g., `counter += 1`, to ensure that it can eventually exceed the condition and terminate the loop.

